In [7]:
! pip install optuna


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [17]:
# Importing the required libraries
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
import optuna
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import train_test_split

In [18]:
# Import necessary libraries
import optuna
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Load the Pima Indian Diabetes dataset from sklearn
# Note: Scikit-learn's built-in 'load_diabetes' is a regression dataset.
# We will load the actual diabetes dataset from an external source
import pandas as pd

# Load the Pima Indian Diabetes dataset (from UCI repository)
url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"
columns = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI',
           'DiabetesPedigreeFunction', 'Age', 'Outcome']

# Load the dataset
df = pd.read_csv(url, names=columns)

df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [20]:
import numpy as np

# Replace zero values with NaN in columns where zero is not a valid value
cols_with_missing_vals = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
df[cols_with_missing_vals] = df[cols_with_missing_vals].replace(0, np.nan)

# Impute the missing values with the mean of the respective column
df.fillna(df.mean(), inplace=True)

# Check if there are any remaining missing values
print(df.isnull().sum())


Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64


In [21]:
# Split into features (X) and target (y)
X = df.drop('Outcome', axis=1)
y = df['Outcome']

# Split data into training and test sets (70% train, 30% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Optional: Scale the data for better model performance
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Check the shape of the data
print(f'Training set shape: {X_train.shape}')
print(f'Test set shape: {X_test.shape}')


Training set shape: (537, 8)
Test set shape: (231, 8)


In [22]:
# Define the objective function for Optuna
def objective(trial):
    # Choose the algorithm to tune
    classifier_name = trial.suggest_categorical('classifier', ['SVM', 'RandomForest', 'GradientBoosting'])

    if classifier_name == 'SVM':
        # SVM hyperparameters
        c = trial.suggest_float('C', 0.1, 100, log=True)
        kernel = trial.suggest_categorical('kernel', ['linear', 'rbf', 'poly', 'sigmoid'])
        gamma = trial.suggest_categorical('gamma', ['scale', 'auto'])

        model = SVC(C=c, kernel=kernel, gamma=gamma, random_state=42)

    elif classifier_name == 'RandomForest':
        # Random Forest hyperparameters
        n_estimators = trial.suggest_int('n_estimators', 50, 300)
        max_depth = trial.suggest_int('max_depth', 3, 20)
        min_samples_split = trial.suggest_int('min_samples_split', 2, 10)
        min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 10)
        bootstrap = trial.suggest_categorical('bootstrap', [True, False])

        model = RandomForestClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            bootstrap=bootstrap,
            random_state=42
        )

    elif classifier_name == 'GradientBoosting':
        # Gradient Boosting hyperparameters
        n_estimators = trial.suggest_int('n_estimators', 50, 300)
        learning_rate = trial.suggest_float('learning_rate', 0.01, 0.3, log=True)
        max_depth = trial.suggest_int('max_depth', 3, 20)
        min_samples_split = trial.suggest_int('min_samples_split', 2, 10)
        min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 10)

        model = GradientBoostingClassifier(
            n_estimators=n_estimators,
            learning_rate=learning_rate,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            random_state=42
        )

    # Perform cross-validation and return the mean accuracy
    score = cross_val_score(model, X_train, y_train, cv=3, scoring='accuracy').mean()
    return score

In [23]:
# Create a study and optimize it using CmaEsSampler
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=100)

[I 2026-08-21 17:20:14,802] A new study created in memory with name: no-name-c5597c2f-1573-408a-99be-4def2abf909c
[I 2026-08-21 17:20:16,175] Trial 0 finished with value: 0.7672253258845437 and parameters: {'classifier': 'RandomForest', 'n_estimators': 182, 'max_depth': 14, 'min_samples_split': 2, 'min_samples_leaf': 4, 'bootstrap': False}. Best is trial 0 with value: 0.7672253258845437.
[I 2026-08-21 17:20:16,236] Trial 1 finished with value: 0.7523277467411545 and parameters: {'classifier': 'SVM', 'C': 0.22095320262992701, 'kernel': 'rbf', 'gamma': 'auto'}. Best is trial 0 with value: 0.7672253258845437.
[I 2026-08-21 17:20:16,395] Trial 2 finished with value: 0.7858472998137801 and parameters: {'classifier': 'SVM', 'C': 29.936464337390817, 'kernel': 'linear', 'gamma': 'auto'}. Best is trial 2 with value: 0.7858472998137801.
[I 2026-08-21 17:20:16,469] Trial 3 finished with value: 0.7076350093109869 and parameters: {'classifier': 'SVM', 'C': 19.22751274323128, 'kernel': 'poly', 'gamm

In [25]:
# Retrieve the best trial
best_trial = study.best_trial
print("Best trial parameters:", best_trial.params)
print("Best trial accuracy:", best_trial.value)

Best trial parameters: {'classifier': 'SVM', 'C': 0.12157857907757849, 'kernel': 'linear', 'gamma': 'scale'}
Best trial accuracy: 0.7895716945996275


In [26]:
study.trials_dataframe()['params_classifier'].value_counts()

params_classifier
SVM                 79
RandomForest        11
GradientBoosting    10
Name: count, dtype: int64

In [30]:
import optuna
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.datasets import load_iris
from sklearn.metrics import accuracy_score
from sklearn.model_selection import cross_val_score
import numpy as np

# Load the Iris dataset
X, y = load_iris(return_X_y=True)

# Split the dataset into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Define the objective function for XGBoost
def objective(trial):
    # Hyperparameter search space
    param = {
        'verbosity': 0,
        'objective': 'multi:softprob',
        'num_class': 3,
        'eval_metric': 'mlogloss',  # Ensure that the eval_metric is specified here
        'booster': 'gbtree',
        'lambda': trial.suggest_float('lambda', 1e-8, 1.0, log=True),
        'alpha': trial.suggest_float('alpha', 1e-8, 1.0, log=True),
        'eta': trial.suggest_float('eta', 0.01, 0.3),
        'gamma': trial.suggest_float('gamma', 1e-8, 1.0, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 9),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'subsample': trial.suggest_float('subsample', 0.4, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.4, 1.0),
        'n_estimators': 300,
    }

    # Create DMatrix for XGBoost
    dtrain = xgb.DMatrix(X_train, label=y_train)
    dtest = xgb.DMatrix(X_test, label=y_test)

    # Define a pruning callback based on evaluation metrics
    pruning_callback = optuna.integration.XGBoostPruningCallback(trial, "eval-mlogloss")  # Match the metric name in the evals list

    # Train the model
    bst = xgb.train(
        param,
        dtrain,
        num_boost_round=300,
        evals=[(dtrain, "train"), (dtest, "eval")],  # Ensure the eval datasets and names are specified
        early_stopping_rounds=30,
        callbacks=[pruning_callback]
    )

    # Predict on the test set
    preds = bst.predict(dtest)
    best_preds = [int(np.argmax(line)) for line in preds]

    # Return accuracy as the objective value
    accuracy = accuracy_score(y_test, best_preds)
    return accuracy

# Create a study with pruning
study = optuna.create_study(direction='maximize', pruner=optuna.pruners.SuccessiveHalvingPruner())
study.optimize(objective, n_trials=50)

# Output the best trial
print(f"Best trial: {study.best_trial.params}")
print(f"Best accuracy: {study.best_value}")


[I 2026-08-21 17:23:19,250] A new study created in memory with name: no-name-e57055b2-7e2c-4a46-81b1-23f3e303c089


[0]	train-mlogloss:0.81255	eval-mlogloss:0.80958
[1]	train-mlogloss:0.61850	eval-mlogloss:0.60184
[2]	train-mlogloss:0.47998	eval-mlogloss:0.44977
[3]	train-mlogloss:0.38696	eval-mlogloss:0.35103
[4]	train-mlogloss:0.31356	eval-mlogloss:0.26984
[5]	train-mlogloss:0.26187	eval-mlogloss:0.21516
[6]	train-mlogloss:0.22470	eval-mlogloss:0.17434
[7]	train-mlogloss:0.19971	eval-mlogloss:0.15201
[8]	train-mlogloss:0.16932	eval-mlogloss:0.12149
[9]	train-mlogloss:0.14942	eval-mlogloss:0.09996
[10]	train-mlogloss:0.13525	eval-mlogloss:0.08888
[11]	train-mlogloss:0.12201	eval-mlogloss:0.07400
[12]	train-mlogloss:0.11278	eval-mlogloss:0.06523
[13]	train-mlogloss:0.10635	eval-mlogloss:0.05885
[14]	train-mlogloss:0.10504	eval-mlogloss:0.05728
[15]	train-mlogloss:0.10004	eval-mlogloss:0.05058
[16]	train-mlogloss:0.09745	eval-mlogloss:0.04761
[17]	train-mlogloss:0.09556	eval-mlogloss:0.04598
[18]	train-mlogloss:0.09461	eval-mlogloss:0.04425
[19]	train-mlogloss:0.09407	eval-mlogloss:0.04364


c:\Users\vinay kumar\anaconda3\Lib\site-packages\optuna\integration\xgboost.py:14: FutureWarning: `optuna.integration.xgboost` has been deprecated in v4.9.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v4.9.0. Use `optuna_integration.xgboost` instead.
  optuna_warn(f"{msg} Use `optuna_integration.xgboost` instead.", FutureWarning)


[20]	train-mlogloss:0.09114	eval-mlogloss:0.04338
[21]	train-mlogloss:0.08920	eval-mlogloss:0.03989
[22]	train-mlogloss:0.08854	eval-mlogloss:0.03809
[23]	train-mlogloss:0.08601	eval-mlogloss:0.03839
[24]	train-mlogloss:0.08580	eval-mlogloss:0.03816
[25]	train-mlogloss:0.08523	eval-mlogloss:0.03674
[26]	train-mlogloss:0.08439	eval-mlogloss:0.03596
[27]	train-mlogloss:0.08369	eval-mlogloss:0.03555
[28]	train-mlogloss:0.08302	eval-mlogloss:0.03475
[29]	train-mlogloss:0.08323	eval-mlogloss:0.03445
[30]	train-mlogloss:0.08270	eval-mlogloss:0.03509
[31]	train-mlogloss:0.08177	eval-mlogloss:0.03483
[32]	train-mlogloss:0.08167	eval-mlogloss:0.03501
[33]	train-mlogloss:0.08206	eval-mlogloss:0.03606
[34]	train-mlogloss:0.08209	eval-mlogloss:0.03608
[35]	train-mlogloss:0.08202	eval-mlogloss:0.03482
[36]	train-mlogloss:0.08165	eval-mlogloss:0.03542
[37]	train-mlogloss:0.08099	eval-mlogloss:0.03516
[38]	train-mlogloss:0.07998	eval-mlogloss:0.03664
[39]	train-mlogloss:0.07979	eval-mlogloss:0.03666


[I 2026-08-21 17:23:19,968] Trial 0 finished with value: 1.0 and parameters: {'lambda': 1.0734299343955154e-06, 'alpha': 1.1138083260149617e-06, 'eta': 0.24705898309327518, 'gamma': 1.914120436104787e-05, 'max_depth': 4, 'min_child_weight': 2, 'subsample': 0.6333438170995954, 'colsample_bytree': 0.5817189906662737}. Best is trial 0 with value: 1.0.


...

[0]	train-mlogloss:0.82766	eval-mlogloss:0.85761


[I 2026-08-21 17:23:26,522] Trial 48 pruned. Trial was pruned at iteration 1.


[0]	train-mlogloss:0.88620	eval-mlogloss:0.90324


[I 2026-08-21 17:23:26,559] Trial 49 pruned. Trial was pruned at iteration 1.


Best trial: {'lambda': 1.0734299343955154e-06, 'alpha': 1.1138083260149617e-06, 'eta': 0.24705898309327518, 'gamma': 1.914120436104787e-05, 'max_depth': 4, 'min_child_weight': 2, 'subsample': 0.6333438170995954, 'colsample_bytree': 0.5817189906662737}
Best accuracy: 1.0


In [29]:
! pip install optuna-integration[xgboost]


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [31]:
from optuna.visualization import plot_intermediate_values

# 1. Plot intermediate values during the trials
plot_intermediate_values(study).show()